# KL vs VQ Autoencoder Evaluation

## Experiment 02: First-Stage Autoencoder Comparison for LDM-Readiness

**Purpose**: Compare trained KL and VQ autoencoders on identical validation data (seed=42) to select the best first-stage representation for Latent Diffusion Model Stage 2.

**Evaluation Focus**: LDM-readiness (reconstruction quality, latent properties, boundary preservation)

**Execution Modes**:
- Sanity Check: 20 validation images (~2-3 minutes)
- Full Evaluation: ~200 validation images (~15-20 minutes)

**Inputs**:
- KL Checkpoint: `/kaggle/input/kl-autoencoder-checkpoints/best_model.pth`
- VQ Checkpoint: `/kaggle/input/vq-autoencoder-checkpoints/best_model.pth`
- Dataset: `/kaggle/input/datasets/tejabulusu/no-mixed/No_mixed`

**Outputs**:
- `/kaggle/working/results/experiment02/kl/` - KL evaluation results
- `/kaggle/working/results/experiment02/vq/` - VQ evaluation results
- `/kaggle/working/results/experiment02/comparison/` - Comparison report

## 1. Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path
import subprocess

# Check environment
print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)
print(f"Python: {sys.version}")
print(f"Working directory: {os.getcwd()}")

# Check GPU
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
print("=" * 70)

## 2. Clone and Setup Repository

In [ ]:
# Clone repository (if not already cloned)
repo_dir = Path('/kaggle/working/Mixed-Pixels')

if not repo_dir.exists():
    print("Cloning repository...")
    subprocess.run([
        'git', 'clone',
        'https://github.com/yourusername/Mixed-Pixels.git',  # Replace with actual repo URL
        str(repo_dir)
    ], check=True)
    print("Repository cloned successfully")
else:
    print("Repository already exists")

# Add to Python path
sys.path.insert(0, str(repo_dir / 'src'))
print(f"\nAdded to path: {repo_dir / 'src'}")

## 3. Verify Checkpoints and Dataset

In [ ]:
# Check checkpoints
kl_checkpoint = Path('/kaggle/input/kl-autoencoder-checkpoints/best_model.pth')
vq_checkpoint = Path('/kaggle/input/vq-autoencoder-checkpoints/best_model.pth')
dataset_dir = Path('/kaggle/input/datasets/tejabulusu/no-mixed/No_mixed')

print("\nVerifying inputs...")
print(f"KL checkpoint exists: {kl_checkpoint.exists()}")
print(f"VQ checkpoint exists: {vq_checkpoint.exists()}")
print(f"Dataset exists: {dataset_dir.exists()}")

if dataset_dir.exists():
    image_files = list(dataset_dir.glob('**/*.png')) + list(dataset_dir.glob('**/*.jpg'))
    print(f"Total images found: {len(image_files)}")

assert kl_checkpoint.exists(), "KL checkpoint not found!"
assert vq_checkpoint.exists(), "VQ checkpoint not found!"
assert dataset_dir.exists(), "Dataset not found!"
print("\n✓ All inputs verified")

## 4. Run Evaluation

### Choose Mode:
- Set `SANITY_CHECK = True` for quick 20-image test
- Set `SANITY_CHECK = False` for full ~200-image evaluation

In [ ]:
# Configuration
SANITY_CHECK = True  # Set to False for full evaluation

print("=" * 70)
print(f"MODE: {'SANITY CHECK (20 images)' if SANITY_CHECK else 'FULL EVALUATION (~200 images)'}")
print("=" * 70)

In [ ]:
# Build command
eval_script = repo_dir / 'scripts' / 'evaluation' / 'evaluate_kl_vs_vq.py'

cmd = [
    'python', str(eval_script),
    '--kl_checkpoint', str(kl_checkpoint),
    '--vq_checkpoint', str(vq_checkpoint),
    '--data_dir', str(dataset_dir),
    '--result_dir', '/kaggle/working/results/experiment02',
    '--batch_size', '8'
]

if SANITY_CHECK:
    cmd.append('--sanity')

print(f"\nRunning: {' '.join(cmd)}\n")
print("=" * 70)

# Run evaluation
result = subprocess.run(cmd, capture_output=False, text=True)

print("=" * 70)
if result.returncode == 0:
    print("\n✓ Evaluation completed successfully!")
else:
    print(f"\n✗ Evaluation failed with return code {result.returncode}")
    raise RuntimeError("Evaluation failed")

## 5. Review Results

In [ ]:
import json

result_dir = Path('/kaggle/working/results/experiment02')

# Load comparison report
comparison_file = result_dir / 'comparison' / 'kl_vs_vq_comparison.json'

if comparison_file.exists():
    with open(comparison_file, 'r') as f:
        comparison = json.load(f)
    
    print("=" * 70)
    print("COMPARISON SUMMARY")
    print("=" * 70)
    
    summary = comparison.get('summary', {})
    
    print(f"\nEvaluation Type: {summary.get('evaluation_type', 'N/A')}")
    print(f"Total Samples: {summary.get('total_samples', 'N/A')}")
    
    print("\nKey Findings:")
    for finding in summary.get('key_findings', []):
        print(f"  • {finding}")
    
    print("\nRecommendation:")
    rec = summary.get('recommendation', {})
    for consideration in rec.get('considerations', []):
        print(f"  • {consideration}")
    
    print(f"\nPreliminary Suggestion: {rec.get('preliminary_suggestion', 'N/A')}")
    print("\n" + "=" * 70)
    
    print("\nNote: Final architectural decision is made by the user.")
    print("This is a technical assessment for LDM Stage 2 selection.")
else:
    print(f"Comparison file not found: {comparison_file}")

## 6. Quick Metrics Display

In [ ]:
# Load individual results
kl_results_file = result_dir / 'kl' / 'kl_evaluation_results.json'
vq_results_file = result_dir / 'vq' / 'vq_evaluation_results.json'

if kl_results_file.exists() and vq_results_file.exists():
    with open(kl_results_file, 'r') as f:
        kl_results = json.load(f)
    with open(vq_results_file, 'r') as f:
        vq_results = json.load(f)
    
    print("\n" + "=" * 70)
    print("RECONSTRUCTION METRICS")
    print("=" * 70)
    
    kl_recon = kl_results.get('reconstruction_metrics', {})
    vq_recon = vq_results.get('reconstruction_metrics', {})
    
    print(f"\nMSE:")
    print(f"  KL: {kl_recon.get('mse', {}).get('mean', 'N/A'):.6f}")
    print(f"  VQ: {vq_recon.get('mse', {}).get('mean', 'N/A'):.6f}")
    
    print(f"\nSSIM:")
    print(f"  KL: {kl_recon.get('ssim', {}).get('mean', 'N/A'):.4f}")
    print(f"  VQ: {vq_recon.get('ssim', {}).get('mean', 'N/A'):.4f}")
    
    print("\n" + "=" * 70)
    print("LDM READINESS")
    print("=" * 70)
    
    kl_ldm = kl_results.get('ldm_readiness', {})
    vq_ldm = vq_results.get('ldm_readiness', {})
    
    print(f"\nReadiness Score:")
    print(f"  KL: {kl_ldm.get('readiness_score', 'N/A'):.3f}")
    print(f"  VQ: {vq_ldm.get('readiness_score', 'N/A'):.3f}")
    
    print(f"\nInterpretation:")
    print(f"  KL: {kl_ldm.get('interpretation', 'N/A')}")
    print(f"  VQ: {vq_ldm.get('interpretation', 'N/A')}")
    
    print("\n" + "=" * 70)
else:
    print("Individual result files not found")

## 7. Export Results

Results are in `/kaggle/working/results/experiment02/`:
- `kl/kl_evaluation_results.json` - KL metrics
- `vq/vq_evaluation_results.json` - VQ metrics  
- `comparison/kl_vs_vq_comparison.json` - Full comparison report
- `*.npy` files - Diagnostic data for visualization

**Next Steps**:
1. If sanity check: Review results, then run full evaluation
2. If full evaluation: Download results to local machine
3. Review with architect agent
4. Make LDM Stage 2 selection decision

In [ ]:
# List all output files
print("\nGenerated files:")
print("=" * 70)

for root, dirs, files in os.walk(result_dir):
    level = root.replace(str(result_dir), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{Path(root).name}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        file_path = Path(root) / file
        size = file_path.stat().st_size
        print(f"{subindent}{file} ({size:,} bytes)")

print("\n" + "=" * 70)
print("✓ Evaluation complete. Results ready for download.")